# 6-31G → def2-TZVP Molecular-Orbital Projection Tool for GAMESS

**Purpose.** Generate a def2-TZVP initial guess (GAMESS `$VEC` for `MOREAD`) by projecting
a converged 6-31G molecular-orbital set onto the def2-TZVP basis. This steers the def2-TZVP
SCF toward the physically correct electronic state and avoids convergence onto spurious
solutions (e.g., open-shell orbitals localizing on ligands instead of the metal).

**Method.** For a fixed nuclear geometry, the 6-31G MO coefficients `C_s` are projected onto
the def2-TZVP basis by least-squares fitting of each MO:

$$ C_{\mathrm{def2}} = S_{\mathrm{def2}}^{-1}\, S_{\mathrm{cross}}\, C_{\mathrm{6\text{-}31G}} $$

where `S_def2` is the def2-TZVP overlap matrix and `S_cross` is the cross-overlap between the
def2-TZVP and 6-31G basis functions. All matrices are evaluated in a unit-normalized Cartesian
representation and reordered between the GAMESS and PySCF basis-function conventions.

**Validation built in.** Before projecting, the tool verifies that the input 6-31G `$VEC` is
orthonormal under the reconstructed overlap ($\lVert C^{\mathsf T} S\, C - I\rVert_\infty$ below a
threshold). This confirms that the GAMESS↔PySCF basis-ordering and normalization maps are
correct for every element present, and halts otherwise.

**Scope of this version.** 6-31G → def2-TZVP, Cartesian Gaussians (6d, 10f). Reusable for any
geometry/conformer with the same element set. For a new element, add its `ATOMIC BASIS SET`
block; the built-in check will stop if the mapping is not valid.

---
### License
MIT License — Copyright (c) 2026 Hiroshi Sakiyama. See the `LICENSE` cell at the end.

### Citation
If you use this tool, please cite the accompanying paper (details to be completed upon
publication) and, if applicable, the archived code DOI. See the **Citation** cell at the end.

### Dependencies (tested)
Python 3.12, PySCF 2.14.0, NumPy 2.4.4.


## Required input files
1. `COORD` : the `COORDINATES (BOHR)` section from the GAMESS output, saved as text.
2. `VEC6`  : the source 6-31G `$VEC` (a `.dat`/`.vec` file containing `$VEC … $END`).
3. `ABS6`  : the `ATOMIC BASIS SET` section (all elements) from a 6-31G GAMESS output.
4. `ABSD`  : the `ATOMIC BASIS SET` section (all elements) from a def2-TZVP GAMESS output.

`ABS6` and `ABSD` are per-element definitions and can be **reused for all structures with the
same element set**.

## 1. User settings

In [ ]:
# ==== user settings ====
COORD = "coordinates.txt"      # GAMESS COORDINATES (BOHR) section
VEC6  = "input_6-31g.vec"      # source 6-31G $VEC
ABS6  = "abs_6-31g.txt"        # 6-31G ATOMIC BASIS SET section
ABSD  = "abs_def2tzvp.txt"     # def2-TZVP ATOMIC BASIS SET section
OUT   = "projected_def2.vec"   # output file (GAMESS $VEC)

SPIN   = 3     # 2S = number of unpaired electrons (quartet = 3)
CHARGE = 2
ORTHO_THRESHOLD = 1e-3   # max|C^T S C - I| tolerance (typically passes at ~1e-6)


## 2. Library imports and function definitions (normally no edits needed)

In [ ]:
import numpy as np, re
from pyscf import gto

# Elements recognized in coordinate/ABS parsing. Add new elements here as needed.
ELEMS = {'CO','O','N','C','H','CL','F','S','P','BR','FE','NI','MN','CU','ZN'}

def read_gamess_bohr(fn):
    els=[]; C=[]
    for line in open(fn):
        p=line.split()
        if len(p)>=5 and p[0].upper() in ELEMS:
            try: x,y,z=float(p[2]),float(p[3]),float(p[4])
            except: continue
            els.append(p[0].upper()); C.append([x,y,z])
    return els, np.array(C)

def read_vec(fn):
    mos=[]; cur=[]
    for line in open(fn):
        s=line.rstrip('\r\n'); t=s.strip()
        if t in ('$VEC','$END') or t=='': continue
        row=int(s[2:5]); body=s[5:]
        cs=[float(body[j:j+15]) for j in range(0,len(body),15) if body[j:j+15].strip()]
        if row==1:
            if cur: mos.append(cur)
            cur=list(cs)
        else: cur.extend(cs)
    if cur: mos.append(cur)
    return np.array(mos)

# Cartesian component order (GAMESS convention)
NCOMP={'S':['s'],'P':['x','y','z'],'L':['s','x','y','z'],
       'D':['xx','yy','zz','xy','xz','yz'],
       'F':['xxx','yyy','zzz','xxy','xxz','xyy','yyz','xzz','yzz','xyz']}

def parse_abs_by_element(fn):
    # Convert an ATOMIC BASIS SET section into {element: [shell types]},
    # recording only the first occurrence of each element.
    elem_shells={}; curlist=None
    for line in open(fn):
        s=line.rstrip('\r'); st=s.strip()
        if st in ('ATOMIC BASIS SET','----------------') or 'NORMALIZED' in st or 'SHELL TYPE' in st or st=='':
            continue
        toks=st.split()
        if len(toks)==1 and toks[0].upper() in ELEMS:
            e=toks[0].upper()
            if e not in elem_shells:
                elem_shells[e]=[]; curlist=elem_shells[e]
            else:
                curlist=None
            continue
        m=re.match(r'^\s*(\d+)\s+([SPDFL])\s+(\d+)\s+([\d.]+)', s)
        if m and curlist is not None:
            shidx=int(m.group(1)); lt=m.group(2)
            if not curlist or curlist[-1][0]!=shidx:
                curlist.append((shidx,lt))
    return {e:[lt for (idx,lt) in sh] for e,sh in elem_shells.items()}

def gamess_keys_for(els, elem_shells):
    labels=[]
    for ai,e in enumerate(els):
        if e not in elem_shells:
            raise KeyError(f"Element {e} not found in the ABS file. "
                           f"Add this element's ATOMIC BASIS SET block.")
        for lt in elem_shells[e]:
            for comp in NCOMP[lt]:
                if lt=='S': k=(ai,'s','s')
                elif lt=='P': k=(ai,'p',comp)
                elif lt=='L': k=(ai,'s' if comp=='s' else 'p',comp)
                elif lt=='D': k=(ai,'d',comp)
                elif lt=='F': k=(ai,'f',comp)
                labels.append(k)
    return labels

def pyscf_keys(mol):
    keys=[]
    for l in mol.cart_labels():
        m=re.match(r'\s*(\d+)\s+([A-Za-z]+)\s+\d([spdf])([a-z]*)', l)
        ai=int(m.group(1)); lc=m.group(3); comp=m.group(4)
        if lc=='s': comp='s'
        keys.append((ai,lc,comp))
    return keys

def with_occ(keys):
    seen={}; out=[]
    for k in keys:
        seen[k]=seen.get(k,0)+1; out.append((k,seen[k]))
    return out

def build_perm(mol, els, elem_shells):
    # permutation mapping GAMESS AO order -> PySCF AO order
    gk=with_occ(gamess_keys_for(els,elem_shells)); pk=with_occ(pyscf_keys(mol))
    pindex={key:i for i,key in enumerate(pk)}
    return np.array([pindex[k] for k in gk])

def write_gamess_vec(C, fn):
    nmo,nao=C.shape
    with open(fn,'w') as f:
        f.write(' $VEC   \n')
        for i in range(nmo):
            mo2=(i+1)%100; row=0; j=0
            while j<nao:
                row+=1; chunk=C[i,j:j+5]
                line=f'{mo2:2d}{row:3d}'
                for x in chunk: line+=f'{x: .8E}'
                f.write(line+'\n'); j+=5
        f.write(' $END   \n')


# ---- Build PySCF basis directly from GAMESS ABS (no reliance on built-in basis names) ----
LIDX={'S':0,'P':1,'D':2,'F':3,'G':4}
def parse_abs_full(fn):
    """Parse ABS into {ELEM: [(shelltype,[(exp,c1,c2),...]),...]} with full exponents/coeffs."""
    elems={}; curshells=None; curshell=None
    for line in open(fn):
        s=line.rstrip('\r'); st=s.strip()
        if st in ('ATOMIC BASIS SET','----------------') or 'NORMALIZED' in st or 'SHELL TYPE' in st or st=='':
            continue
        toks=st.split()
        if len(toks)==1 and toks[0].upper() in ELEMS:
            e=toks[0].upper()
            if e not in elems: elems[e]=[]; curshells=elems[e]
            else: curshells=None
            curshell=None; continue
        if curshells is None: continue
        m=re.match(r'^\s*(\d+)\s+([SPDFLG])\s+(\d+)\s+([-\d.]+)\s+([-\d.]+)(?:\s+([-\d.]+))?', s)
        if m:
            shidx=int(m.group(1)); lt=m.group(2); exp=float(m.group(4))
            c1=float(m.group(5)); c2=m.group(6)
            if curshell is None or curshell[0]!=shidx:
                curshell=[shidx,lt,[]]; curshells.append(curshell)
            curshell[2].append((exp,c1,c2))
    return elems

def abs_to_pyscf_basis(elem_shells_full):
    """Convert to PySCF basis format {Elem:[[l,[exp,coef],...],...]}. L split into S+P."""
    basis={}
    for e,shells in elem_shells_full.items():
        blocks=[]
        for (shidx,lt,prims) in shells:
            if lt=='L':
                sblock=[0]; pblock=[1]
                for (exp,c1,c2) in prims:
                    sblock.append([exp,c1]); pblock.append([exp,float(c2)])
                blocks.append(sblock); blocks.append(pblock)
            else:
                blk=[LIDX[lt]]
                for (exp,c1,c2) in prims: blk.append([exp,c1])
                blocks.append(blk)
        basis[e.capitalize()]=blocks
    return basis

def make_mol(atom, els, absfn, spin, charge):
    """Build a PySCF Mole using the basis defined in the GAMESS ABS file."""
    full=parse_abs_full(absfn)
    bdict=abs_to_pyscf_basis(full)
    basis={e.capitalize():bdict[e.capitalize()] for e in set(els)}
    return gto.M(atom=atom, basis=basis, unit='Bohr', cart=True, spin=spin, charge=charge, verbose=0)


## 3. Build molecule and basis maps — with automatic orthonormality check

The self-consistency check runs here. If the reconstructed basis map is not valid
(e.g., an unverified new element), the `assert` stops execution.

In [ ]:
els, C = read_gamess_bohr(COORD)
atom=[[e.capitalize(), tuple(C[i])] for i,e in enumerate(els)]
print(f"Atoms: {len(els)}   Elements: {sorted(set(els))}")

# Build molecules using the basis sets defined in the GAMESS ABS files
# (this avoids any dependence on PySCF built-in basis names such as '6-31g').
mol_s=make_mol(atom, els, ABS6, SPIN, CHARGE)
mol_d=make_mol(atom, els, ABSD, SPIN, CHARGE)
print(f"6-31G nao = {mol_s.nao_cart()}   def2-TZVP nao = {mol_d.nao_cart()}")

ES6=parse_abs_by_element(ABS6)
ESD=parse_abs_by_element(ABSD)

Ss=mol_s.intor('int1e_ovlp_cart'); ds=np.sqrt(Ss.diagonal()); Ss_u=Ss*np.outer(1/ds,1/ds)
Sd=mol_d.intor('int1e_ovlp_cart'); dd=np.sqrt(Sd.diagonal()); Sd_u=Sd*np.outer(1/dd,1/dd)

perm_s=build_perm(mol_s, els, ES6); inv_s=np.argsort(perm_s)
perm_d=build_perm(mol_d, els, ESD); inv_d=np.argsort(perm_d)

Cg_s=read_vec(VEC6); Cs_p=Cg_s[:, inv_s]
M=Cs_p @ Ss_u @ Cs_p.T
err=np.abs(M-np.eye(M.shape[0])).max()
print(f"\n[orthonormality check] 6-31G  max|C^T S C - I| = {err:.2e}")
assert err < ORTHO_THRESHOLD, (
    f"Check FAILED ({err:.2e} > {ORTHO_THRESHOLD:.0e}). The basis map is not valid. "
    f"Verify the ABS block for any new element.")
print("OK: basis map verified. Proceeding to projection.")


## 4. Perform the projection and write the GAMESS `$VEC`

In [ ]:
Scross=gto.intor_cross('int1e_ovlp_cart', mol_d, mol_s)
Scross_u=Scross*np.outer(1/dd, 1/ds)

# C_def2 = S_def2^{-1} S_cross C_6-31G   (unit-normalized, PySCF order)
Cd_p=(np.linalg.inv(Sd_u) @ Scross_u @ Cs_p.T).T

# reorder back to GAMESS AO order and write
Cd_gamess=Cd_p[:, perm_d]
write_gamess_vec(Cd_gamess, OUT)
print(f"Projection complete: wrote {OUT}  ({Cd_gamess.shape[0]} MOs x {Cd_gamess.shape[1]} AOs)")


## 5. Quality checks (optional)

Confirms that the projected MOs preserve the 6-31G occupied space and, for Co complexes,
that the singly occupied MOs (SOMOs) retain metal-d character.

In [ ]:
Z={'CO':27,'O':8,'N':7,'C':6,'H':1,'CL':17,'F':9,'S':16,'P':15,'BR':35,
   'FE':26,'NI':28,'MN':25,'CU':29,'ZN':30}
ne=sum(Z[e] for e in els) - CHARGE; nb=(ne-SPIN)//2; na=nb+SPIN
print(f"Electrons {ne}   doubly occ {nb}   singly occ {SPIN}   highest occ MO {na}")

# occupied-space preservation (6-31G occ vs projected def2 occ)
O=Cd_p[:na] @ Scross_u @ Cs_p[:na].T
sv=np.linalg.svd(O, compute_uv=False)
print(f"Occupied-space preservation: sv min {sv.min():.4f}  mean {sv.mean():.4f}  "
      f">0.95: {(sv>0.95).sum()}/{na}")

# SOMO metal-d character (Co-containing systems only)
labs=mol_d.cart_labels()
co_d=[i for i,l in enumerate(labs) if re.search(r'\bCo\b',l) and re.search(r'd[a-z]{2}',l)]
if co_d:
    print("\nCo-d weight near the SOMO region:")
    for mo in range(na-4, na+1):
        v=Cd_p[mo-1]; contrib=v*(Sd_u@v)
        print(f"  MO {mo}: Co-d weight = {contrib[co_d].sum():.3f}")


## Usage summary

1. Set file paths and `SPIN`/`CHARGE` in Section 1.
2. Run all cells top to bottom.
3. If the orthonormality check in Section 3 passes, use the generated `projected_def2.vec`
   in the GAMESS input via `$GUESS GUESS=MOREAD NORB=... $END`.

**Different structure/conformer:** replace `COORD` and `VEC6` only (reuse the ABS files if the
element set is unchanged).

**New element:** append its ABS block to `ABS6`/`ABSD`, and add it to `ELEMS` (and to `Z` for
the optional quality check). The built-in orthonormality check will pass if the mapping is
correct and stop otherwise.

**Method note for reproducibility.** The projection is a least-squares basis-set projection,
$C_{\mathrm{def2}} = S_{\mathrm{def2}}^{-1} S_{\mathrm{cross}} C_{\mathrm{6\text{-}31G}}$,
evaluated with unit-normalized Cartesian Gaussians. Overlap and cross-overlap integrals are
computed with PySCF; GAMESS↔PySCF basis-function ordering (including L-shell s/p splitting and
Cartesian d/f component order) and per-function normalization are reconciled explicitly and
verified by the orthonormality check.


## License (MIT)

```
MIT License

Copyright (c) 2026 Hiroshi Sakiyama

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```


## Citation

If this tool contributes to your work, please cite:

- **Paper:** Hiroshi Sakiyama et al., *"<title to be completed upon publication>"*,
  <journal>, <year>. DOI: <to be added>.
- **Software (optional):** archived code DOI (e.g., Zenodo): <to be added>.

Please also cite **PySCF**, which this tool depends on:
- Q. Sun et al., "Recent developments in the PySCF program package,"
  *J. Chem. Phys.* **153**, 024109 (2020).
